# Update Analysis CSV

**Source script:** `update_analysis_csv.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path
import sys

import pandas as pd


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Update analysis_dataset.csv from analysis_dataset.xlsx.",
    )
    parser.add_argument(
        "--xlsx",
        default="analysis_dataset.xlsx",
        help="Path to the Excel file (default: analysis_dataset.xlsx).",
    )
    parser.add_argument(
        "--csv",
        default="analysis_dataset.csv",
        help="Path to the output CSV file (default: analysis_dataset.csv).",
    )
    parser.add_argument(
        "--sheet",
        default=None,
        help="Excel sheet name or index to export (default: first sheet). Use 'all' to export all sheets.",
    )
    parser.add_argument(
        "--list-sheets",
        action="store_true",
        help="List available sheet names and exit.",
    )
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    xlsx_path = Path(args.xlsx)
    csv_path = Path(args.csv)

    if not xlsx_path.exists():
        print(f"Missing Excel file: {xlsx_path}", file=sys.stderr)
        return 1

    if args.list_sheets:
        xls = pd.ExcelFile(xlsx_path)
        print("\n".join(xls.sheet_names))
        return 0

    sheet_arg = args.sheet
    if sheet_arg == "all":
        sheets = pd.read_excel(xlsx_path, sheet_name=None)
        out_dir = csv_path.parent
        out_dir.mkdir(parents=True, exist_ok=True)
        for name, sheet_df in sheets.items():
            safe_name = str(name).strip().replace(" ", "_")
            out_path = out_dir / f"{csv_path.stem}__{safe_name}{csv_path.suffix}"
            sheet_df.to_csv(out_path, index=False)
            print(f"Wrote {out_path} with {len(sheet_df)} rows and {len(sheet_df.columns)} columns.")
        return 0

    df = pd.read_excel(xlsx_path, sheet_name=sheet_arg)
    if isinstance(df, dict):

        first_name = next(iter(df.keys()))
        print(f"No sheet specified; defaulting to first sheet: {first_name}")
        df = df[first_name]

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)

    print(f"Wrote {csv_path} with {len(df)} rows and {len(df.columns)} columns.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
